In [10]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchaudio
import snntorch as snn
from snntorch import surrogate
import torch.nn as nn
import torch.nn.functional as F

class ESC50Dataset(Dataset):
    def __init__(self, csv_path, audio_dir, transform=None):
        self.data = pd.read_csv(csv_path)
        self.audio_dir = audio_dir
        self.transform = transform
        self.mel_spec = torchaudio.transforms.MelSpectrogram(sample_rate=44100, n_mels=128)
        self.label2idx = {label: idx for idx, label in enumerate(sorted(self.data["target"].unique()))}

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row["filename"])
        waveform, sr = torchaudio.load(audio_path)
        waveform = waveform.mean(dim=0, keepdim=True)  # mono

        mel = self.mel_spec(waveform)
        mel = torchaudio.functional.amplitude_to_DB(mel, multiplier=10, amin=1e-10, db_multiplier=0)

        if self.transform:
            mel = self.transform(mel)

        label_idx = self.label2idx[row["target"]]
        return mel, torch.tensor(label_idx)

# Update paths
csv_path = '/media/arafat/New Volume/ESC-50-master/meta/esc50.csv'
audio_dir = '/media/arafat/New Volume/ESC-50-master/audio'

# Dataset
full_dataset = ESC50Dataset(csv_path, audio_dir)

# Split (70% train, 20% test, 10% val)
total = len(full_dataset)
train_size = int(0.7 * total)
test_size = int(0.1 * total)
val_size = total - train_size - test_size

train_set, test_set, val_set = random_split(full_dataset, [train_size, test_size, val_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32)
test_loader = DataLoader(test_set, batch_size=32)


/home/arafat/work/thesis/snn/env/lib/python3.10/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [11]:
len(train_loader), len(val_loader), len(test_loader)

(44, 13, 7)

In [12]:
import snntorch.spikeplot as splt

# Spiking activation function
spike_grad = surrogate.fast_sigmoid()

# Define spiking network
class SNNNet(nn.Module):
    def __init__(self, input_size, num_classes, beta=0.9):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True)

        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True)

        self.pool2 = nn.MaxPool2d(2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 32 * 43, 256)  # Adjust based on mel spec dims after pooling
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad, init_hidden=True)

        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x, num_steps=10):
        mem1, spike1 = self.lif1.init_leaky()
        mem2, spike2 = self.lif2.init_leaky()
        mem3, spike3 = self.lif3.init_leaky()

        spk_out = 0

        for step in range(num_steps):
            cur_input = x
            cur_input = self.conv1(cur_input)
            spk1, mem1 = self.lif1(cur_input, mem1)

            x1 = self.pool1(spk1)
            x2 = self.conv2(x1)
            spk2, mem2 = self.lif2(x2, mem2)

            x3 = self.pool2(spk2)
            x4 = self.flatten(x3)
            x5 = self.fc1(x4)
            spk3, mem3 = self.lif3(x5, mem3)

            out = self.fc2(spk3)
            spk_out += out

        return spk_out / num_steps


In [13]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model already created earlier
model = SNNNet(input_size=(1, 128, 172), num_classes=50).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [14]:
from tqdm import tqdm

def train(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]")
        for inputs, labels in loop:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs, num_steps=10)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

            loop.set_postfix(loss=loss.item(), acc=100.*correct/total)

        # Validation
        val_acc, val_loss = evaluate(model, val_loader)
        print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs, num_steps=10)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    accuracy = 100. * correct / total
    avg_loss = total_loss / len(loader)
    return accuracy, avg_loss


In [15]:
def test(model, test_loader):
    test_acc, test_loss = evaluate(model, test_loader)
    print(f"\nTest Accuracy: {test_acc:.2f}%, Loss: {test_loss:.4f}")


In [16]:
train(model, train_loader, val_loader, epochs=10)
test(model, test_loader)


Epoch [1/10]:   0%|          | 0/44 [00:00<?, ?it/s]


ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
# Set device with memory management
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Clear GPU cache at start
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.set_per_process_memory_fraction(0.9)  # Use 90% of available GPU memory
    print(
        f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB"
    )

# Network Architecture - Optimized for 64x64 RGB images
num_inputs = 28 * 28 * 3
num_hidden = 512
num_outputs = 50

# Temporal Dynamics - Optimized
num_steps = 25
beta = 0.95


# Define Optimized Network
class OptimizedNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(num_inputs, num_hidden, bias=False)
        self.bn1 = nn.BatchNorm1d(num_hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid(slope=25))

        self.fc2 = nn.Linear(
            num_hidden, num_hidden // 2, bias=False
        )  # Additional layer for better feature extraction
        self.bn2 = nn.BatchNorm1d(num_hidden // 2)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=surrogate.fast_sigmoid(slope=25))

        self.fc3 = nn.Linear(num_hidden // 2, num_outputs, bias=False)
        self.lif3 = snn.Leaky(
            beta=beta, spike_grad=surrogate.fast_sigmoid(slope=25), output=True
        )

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        batch_size = x.size(0)

        # Flatten the input images
        x = x.view(batch_size, -1)

        # Initialize hidden states at t=0
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        # Memory-efficient spike accumulation instead of storing all timesteps
        spike_sum = torch.zeros(batch_size, num_outputs, device=x.device)

        for step in range(num_steps):
            # First layer
            cur1 = self.bn1(self.fc1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            spk1 = self.dropout(spk1)

            # Second layer
            cur2 = self.bn2(self.fc2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            spk2 = self.dropout(spk2)

            # Output layer
            cur3 = self.fc3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)

            # Accumulate spikes instead of storing all timesteps
            spike_sum += spk3

            # Clean up intermediate tensors for memory efficiency
            del cur1, spk1, cur2, spk2, cur3, spk3

        return spike_sum


# Load the network onto CUDA if available
net = OptimizedNet().to(device)

# Print model info
total_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print("Model Architecture:")
print(net)

# Optimized loss function and optimizer
loss_fn = SF.ce_rate_loss()
optimizer = torch.optim.AdamW(net.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# Mixed precision scaler for memory and speed optimization
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None


def train_model(net, train_loader, val_loader, num_epochs=100, patience=15):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    best_val_acc = 0.0
    patience_counter = 0

    print("Starting optimized SNN training...")

    for epoch in range(num_epochs):
        # Training phase
        net.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for batch_idx, (data, targets) in enumerate(train_bar):
            data, targets = data.to(device, non_blocking=True), targets.to(
                device, non_blocking=True
            )

            optimizer.zero_grad()

            # Mixed precision forward pass
            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = net(data)
                    loss = loss_fn(
                        outputs.unsqueeze(0), targets
                    )  # Add time dimension for loss

                # Mixed precision backward pass
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = net(data)
                loss = loss_fn(outputs.unsqueeze(0), targets)
                loss.backward()
                optimizer.step()

            # Statistics
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train += targets.size(0)
            correct_train += (predicted == targets).sum().item()

            # Memory management
            if batch_idx % 10 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

            train_bar.set_postfix(
                {
                    "Loss": f"{loss.item():.4f}",
                    "Acc": f"{100.*correct_train/total_train:.2f}%",
                }
            )

            del data, targets, outputs, loss

        # Validation phase
        net.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for data, targets in val_bar:
                data, targets = data.to(device, non_blocking=True), targets.to(
                    device, non_blocking=True
                )

                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = net(data)
                        loss = loss_fn(outputs.unsqueeze(0), targets)
                else:
                    outputs = net(data)
                    loss = loss_fn(outputs.unsqueeze(0), targets)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += targets.size(0)
                correct_val += (predicted == targets).sum().item()

                val_bar.set_postfix(
                    {
                        "Loss": f"{loss.item():.4f}",
                        "Acc": f"{100.*correct_val/total_val:.2f}%",
                    }
                )

                del data, targets, outputs, loss

        # Calculate epoch metrics
        epoch_train_loss = running_loss / len(train_loader)
        epoch_val_loss = val_loss / len(val_loader)
        epoch_train_acc = 100.0 * correct_train / total_train
        epoch_val_acc = 100.0 * correct_val / total_val

        train_losses.append(epoch_train_loss)
        val_losses.append(epoch_val_loss)
        train_accuracies.append(epoch_train_acc)
        val_accuracies.append(epoch_val_acc)

        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%")
        print(f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")

        if torch.cuda.is_available():
            print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        print("-" * 60)

        scheduler.step()

        # Early stopping and model saving
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            patience_counter = 0
            torch.save(net.state_dict(), "best_simple_snn_model.pth")
            print(f"New best validation accuracy: {best_val_acc:.2f}%")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

        # Clear cache after each epoch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return train_losses, val_losses, train_accuracies, val_accuracies


def test_model(net, test_loader):
    net.eval()
    correct = 0
    total = 0
    class_correct = list(0.0 for i in range(num_outputs))
    class_total = list(0.0 for i in range(num_outputs))

    with torch.no_grad():
        test_bar = tqdm(test_loader, desc="Testing")
        for data, targets in test_bar:
            data, targets = data.to(device, non_blocking=True), targets.to(
                device, non_blocking=True
            )

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = net(data)
            else:
                outputs = net(data)

            _, predicted = torch.max(outputs, 1)

            total += targets.size(0)
            correct += (predicted == targets).sum().item()

            # Per-class accuracy
            c = (predicted == targets).squeeze()
            for i in range(targets.size(0)):
                label = targets[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1

            test_bar.set_postfix({"Acc": f"{100.*correct/total:.2f}%"})

            del data, targets, outputs, predicted

    test_accuracy = 100.0 * correct / total
    print(f"\nTest Accuracy: {test_accuracy:.2f}%")

    # Print per-class accuracy
    class_names = [
        "dog",
        "rooster",
        "pig",
        "cow",
        "frog",
        "cat",
        "hen",
        "insects",
        "sheep",
        "crow",
        "rain",
        "sea_waves",
        "crackling_fire",
        "crickets",
        "chirping_birds",
        "water_drops",
        "wind",
        "pouring_water",
        "toilet_flush",
        "thunderstorm",
        "crying_baby",
        "sneezing",
        "clapping",
        "breathing",
        "coughing",
        "footsteps",
        "laughing",
        "brushing_teeth",
        "snoring",
        "drinking_sipping",
        "door_wood_knock",
        "mouse_click",
        "keyboard_typing",
        "door_wood_creaks",
        "can_opening",
        "washing_machine",
        "vacuum_cleaner",
        "clock_alarm",
        "clock_tick",
        "glass_breaking",
        "helicopter",
        "chainsaw",
        "siren",
        "car_horn",
        "engine",
        "train",
        "church_bells",
        "airplane",
        "fireworks",
        "hand_saw",
    ]

    print("\nPer-class accuracy:")
    for i in range(num_outputs):
        if class_total[i] > 0:
            class_acc = 100 * class_correct[i] / class_total[i]
            print(f"{class_names[i]}: {class_acc:.2f}%")

    return test_accuracy


def plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot losses
    ax1.plot(train_losses, label="Training Loss")
    ax1.plot(val_losses, label="Validation Loss")
    ax1.set_title("Training and Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    # Plot accuracies
    ax2.plot(train_accuracies, label="Training Accuracy")
    ax2.plot(val_accuracies, label="Validation Accuracy")
    ax2.set_title("Training and Validation Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig("simple_snn_training_history.png", dpi=300, bbox_inches="tight")
    plt.show()


# Train the model
print("Training Simple SNN model for environmental sound recognition...")
train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    net, train_loader, val_loader, num_epochs=100, patience=15
)

# Load best model and test
net.load_state_dict(torch.load("best_simple_snn_model.pth"))
test_accuracy = test_model(net, test_loader)

# Plot training history
plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies)

print(f"\nFinal Test Accuracy: {test_accuracy:.2f}%")

# Clear GPU memory at the end
if torch.cuda.is_available():
    torch.cuda.empty_cache()